# Étape 04 — jointure des données

Assemble les sorties des étapes 00 à 03 (capteurs, communes, clusters d'usage, classification, réseau
cyclable) en trois tables - le pont vers les étapes 05/06/99, qui ne lisent plus que
`data/donnees_valides/bkt/` (jamais les dossiers des étapes 00-03 directement, voir le README).

Chaque ingrédient est chargé séparément (sections 1-2), avant le merge (section 3) puis l'assemblage
complet des 7 années x 3 scénarios (section 4).

## 0. Configuration

In [1]:
import sys
from pathlib import Path

ICI = Path.cwd()
sys.path.insert(0, str(ICI))
RACINE = ICI.parents[1]
sys.path.insert(0, str(RACINE / "src" / "00_transformation_des_donnees"))
sys.path.insert(0, str(RACINE / "src" / "00_transformation_des_donnees" / "d_zones"))

import pandas as pd
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)

DONNEES_VALIDES = RACINE / "data" / "donnees_valides"
DOSSIER_SORTIE = DONNEES_VALIDES / "bkt"
ANNEES = (2019, 2020, 2021, 2022, 2023, 2024, 2025)

FICHIER_COMMUNES_ZONES = DONNEES_VALIDES / "zones" / "communes.parquet"
FICHIER_SENSOR_YEARS = DONNEES_VALIDES / "capteurs" / "sensor_years.parquet"
FICHIER_CLUSTER_ASSIGNMENTS = DONNEES_VALIDES / "clustering" / "cluster_assignments.parquet"
FICHIER_COMMUNE_CLUSTERS = DONNEES_VALIDES / "classification" / "commune_clusters.parquet"
DOSSIER_RESEAU_FOB = DONNEES_VALIDES / "reseau_fob"
FICHIER_LONGUEURS_REFERENCE = RACINE / "data" / "donnees_brutes" / "reference" / "longueurs_reseau_reference.csv"

## 1. Les ingrédients, chargés séparément — [`entrees.py`](entrees.py)

`capteurs_actifs()`, `table_communes()` et `communes_instrumentees()` viennent des étapes 00 à 02 -
rien à recalculer ici, le merge n'arrive qu'en section 3.

In [2]:
from entrees import capteurs_actifs, communes_instrumentees, etendre_arrondissements_plm, table_communes

capteurs = capteurs_actifs(FICHIER_SENSOR_YEARS, FICHIER_CLUSTER_ASSIGNMENTS)
print(f"\n{len(capteurs):,} lignes (capteur, année) - pas de réseau ni de population ici, juste débit + cluster :")
capteurs[["id_site", "annee", "id_commune_str", "annual_flow", "qta", "cluster_K4"]].head(3)

[entrees] 7,852 capteurs-années actifs ; 339 sans cluster K4 (id_site absent de la référence 01)

7,852 lignes (capteur, année) - pas de réseau ni de population ici, juste débit + cluster :


,id_site,annee,id_commune_str,annual_flow,qta,cluster_K4
0,200000252,2019,22371,6394.0,6967.463615,3.0
1,200049684,2019,02751,11748.0,14587.169383,3.0
2,200049685,2019,02490,10715.0,12339.082424,3.0


In [3]:
communes = table_communes(FICHIER_COMMUNES_ZONES, FICHIER_COMMUNE_CLUSTERS)
print(f"{len(communes):,} communes - aucune notion de réseau, de population, ni d'année ici (une seule ligne par commune, pas par année) :")
communes.head(3)

34,428 communes - aucune notion de réseau, de population, ni d'année ici (une seule ligne par commune, pas par année) :


,code_commune,nom_commune,code_departement,cluster_K4
0,01001,L'Abergement-Clémenciat,01,2
1,01002,L'Abergement-de-Varey,01,2
2,01004,Ambérieu-en-Bugey,01,0


### Extension PLM — [`etendre_arrondissements_plm`](entrees.py)

Les capteurs parisiens de ce jeu de données signalent leur commune sous le code de la ville entière
(`75056`), pas sous un arrondissement (`751xx`) - sans extension, aucun des 20 arrondissements ne
serait compté comme instrumenté.

In [4]:
sensor_years = pd.read_parquet(FICHIER_SENSOR_YEARS, columns=["id_commune_str", "active"])
codes_actifs_paris = {c for c in sensor_years.loc[sensor_years["active"], "id_commune_str"].dropna() if c.startswith("75")}
arrondissements_751xx_avant = {c for c in codes_actifs_paris if c.startswith("751")}
instrumentees = communes_instrumentees(FICHIER_SENSOR_YEARS)
paris_avec_extension = {c for c in instrumentees if c.startswith("751")}

print(f"codes actifs commençant par '75' : {sorted(codes_actifs_paris)} - la ville entière, pas un arrondissement")
print(f"arrondissements 751xx qui matcheraient sans extension : {len(arrondissements_751xx_avant)}")
print(f"arrondissements 751xx comptés instrumentés après extension : {len(paris_avec_extension)} sur 20")
print(f"\n{len(instrumentees):,} communes instrumentées au total (toutes années confondues)")

codes actifs commençant par '75' : ['75056'] - la ville entière, pas un arrondissement
arrondissements 751xx qui matcheraient sans extension : 0
arrondissements 751xx comptés instrumentés après extension : 20 sur 20

1,023 communes instrumentées au total (toutes années confondues)


## 2. Réseau cyclable et population — [`longueurs.py`](longueurs.py) / [`assemblage.table_population_utilisee`](assemblage.py)

In [5]:
from longueurs import SCENARIOS, charger_longueurs

longueurs = charger_longueurs(FICHIER_LONGUEURS_REFERENCE, DOSSIER_RESEAU_FOB, ANNEES, source="reference")
print(f"\n{len(longueurs):,} lignes (année, scénario, commune) - {SCENARIOS} x {len(ANNEES)} années :")
longueurs.head(3)

[longueurs] référence : 7 années, 275,046 lignes (scenario, commune)

275,046 lignes (année, scénario, commune) - ('GV', 'FOB', 'FOB_AM') x 7 années :


,annee,code_commune,len_d,len_g,scenario
0,2019,01004,7.3767,7.3767,GV
1,2019,01007,2.7404,2.7404,GV
2,2019,01008,0.0911,0.0911,GV


In [6]:
from assemblage import table_population_utilisee
from population import charger_population

populations, population_utilisee = table_population_utilisee(ANNEES, lambda millesime: charger_population(DONNEES_VALIDES / "zones", millesime))
population_utilisee

,annee,millesime_utilise,regle,population_totale,n_communes
0,2019,2019,année,68229198.0,34990
1,2020,2020,année,68388307.0,34980
2,2021,2021,année,68620565.0,34970
3,2022,2022,année,69009539.0,34964
4,2023,2023,année,69342437.0,34904
5,2024,2023,plus proche avant (aucun millésime plus récent...,69342437.0,34904
6,2025,2023,plus proche avant (aucun millésime plus récent...,69342437.0,34904


## 3. Le merge, sur un (année, scénario) — [`assemblage.base_par_commune`](assemblage.py)

`base_par_commune()` fusionne `communes` avec le réseau et la population d'une année/scénario donnés -
LEFT JOIN, chaque commune est gardée même sans réseau (`fillna(0.0)`).

In [7]:
print("colonnes de 'communes' avant le merge :", list(communes.columns))

colonnes de 'communes' avant le merge : ['code_commune', 'nom_commune', 'code_departement', 'cluster_K4']


In [8]:
from assemblage import base_par_commune

base_2025_fob = base_par_commune(communes, longueurs, annee=2025, scenario="FOB", population=populations[2023])
print("colonnes de 'base_par_commune' après le merge :", list(base_2025_fob.columns))
print(f"\n{len(base_2025_fob):,} lignes (inchangé : LEFT JOIN, chaque commune est gardée même sans réseau, fillna(0.0))")
print(f"communes avec du réseau FOB 2025 : {(base_2025_fob['total_length_km'] > 0).sum():,} / {len(base_2025_fob):,}")
base_2025_fob.sort_values("total_length_km", ascending=False)[["code_commune", "nom_commune", "total_length_km", "population"]].head(5)

colonnes de 'base_par_commune' après le merge : ['code_commune', 'nom_commune', 'code_departement', 'cluster_K4', 'len_d', 'len_g', 'total_length_km', 'population']

34,428 lignes (inchangé : LEFT JOIN, chaque commune est gardée même sans réseau, fillna(0.0))
communes avec du réseau FOB 2025 : 13,523 / 34,428


,code_commune,nom_commune,total_length_km,population
11425,31555,Toulouse,1276.408635,519940.0
15915,44109,Nantes,710.191816,332515.0
26141,67482,Strasbourg,705.078461,296552.0
11983,33063,Bordeaux,632.225287,271552.0
13016,35238,Rennes,520.371809,234950.0


## 4. L'assemblage complet — [`assemblage.assembler`](assemblage.py)

Répète le merge de la section 3 pour les 7 années x 3 scénarios (GV/FOB/FOB_AM), ajoute le statut
"instrumentée" (section 1), et ne garde que les (année, commune) avec du réseau dans au moins un
scénario - d'où le passage de 34 428 communes (niveau commune) à 91 682 lignes (niveau année x commune).

In [9]:
from assemblage import assembler

communes_assemblees = assembler(communes, longueurs, instrumentees, populations, ANNEES, SCENARIOS)
print(f"{len(communes)} communes (niveau : commune) -> {len(communes_assemblees):,} lignes (niveau : année x commune, {communes_assemblees['annee'].nunique()} années)")
print(f"colonnes ajoutées par l'assemblage : {set(communes_assemblees.columns) - set(communes.columns)}")
communes_assemblees.head(3)

34428 communes (niveau : commune) -> 91,682 lignes (niveau : année x commune, 7 années)
colonnes ajoutées par l'assemblage : {'len_d_FOB', 'len_g_FOB', 'len_d_GV', 'annee', 'len_g_FOB_AM', 'population', 'len_g_GV', 'len_d_FOB_AM', 'instrumentee'}


,annee,code_commune,cluster_K4,population,len_d_GV,len_g_GV,len_d_FOB,len_g_FOB,len_d_FOB_AM,len_g_FOB_AM,instrumentee,nom_commune,code_departement
0,2019,01004,0,14514.0,7.3767,7.3767,5.124262,5.124262,5.124262,5.124262,0,Ambérieu-en-Bugey,01
1,2019,01007,2,2915.0,2.7404,2.7404,0.562431,0.562431,0.562431,0.562431,0,Ambronay,01
2,2019,01008,2,777.0,0.0911,0.0911,0.091044,0.091044,0.000000,0.000000,0,Ambutrix,01


## 5. Les trois tables écrites par [`pipeline.py`](pipeline.py)

Vérification : ce notebook doit produire exactement ce que `pipeline.py` écrit.

In [10]:
if not (DOSSIER_SORTIE / "communes.parquet").exists():
    print("Pas encore assembl� par pipeline.py - lancement.")
    import runpy
    runpy.run_path(str(ICI / "pipeline.py"), run_name="__main__")

communes_pipeline = pd.read_parquet(DOSSIER_SORTIE / "communes.parquet")
identique = communes_assemblees.reset_index(drop=True).equals(communes_pipeline.reset_index(drop=True))
print(f"communes.parquet (pipeline.py) == r�sultat de ce notebook : {identique}")
assert identique, "le notebook et pipeline.py divergent - incoh�rence � corriger"

communes.parquet (pipeline.py) == r�sultat de ce notebook : True


## 6. Vérifications de cohérence interne

In [11]:
sys.path.insert(0, str(ICI))
from schemas_jointure import LigneCommuneAssemblee, LigneCapteurAssemble, LignePopulationUtilisee
from schemas import valider_echantillon

sensor_detail = pd.read_parquet(DOSSIER_SORTIE / "sensor_detail.parquet")
valider_echantillon(communes_pipeline, LigneCommuneAssemblee)
valider_echantillon(sensor_detail, LigneCapteurAssemble)
valider_echantillon(pd.read_parquet(DOSSIER_SORTIE / "population_used.parquet"), LignePopulationUtilisee)

[schemas] 20/20 lignes valides pour LigneCommuneAssemblee


[schemas] 20/20 lignes valides pour LigneCapteurAssemble
[schemas] 7/7 lignes valides pour LignePopulationUtilisee


In [12]:
# univers de communes cohérent : toute commune "instrumentée" doit avoir au moins un capteur ACTIF
# associé (PLM compris) - plus d'orphelins depuis cette version (voir le README : ils ne changeaient
# jamais 'bkt_obs', retirés).
codes_actifs = set(sensor_detail["id_commune_str"].dropna())
codes_couverts = etendre_arrondissements_plm(codes_actifs)
communes_instrumentees_table = set(communes_pipeline.loc[communes_pipeline["instrumentee"] == 1, "code_commune"])
manquantes = communes_instrumentees_table - codes_couverts
print(f"communes marquées instrumentées sans capteur actif correspondant : {len(manquantes)}")
assert len(manquantes) == 0, "incohérence : une commune est instrumentée sans capteur actif associé"
print("OK.")

communes marquées instrumentées sans capteur actif correspondant : 0
OK.
